# Notebook — Motor de matching y evaluación (Levenshtein, Jaro-Winkler, Fonético, Embeddings)

Este notebook está pensado para:
1) Cargar `terceros_sinteticos.csv` (base sintética)  
2) Cargar `consolidado` desde SQLite (`analytics.db`)  
3) Construir un set de candidatos (blocking) para evitar comparar contra todo  
4) Ejecutar 4 modelos de similitud:
   - **Levenshtein**
   - **Jaro-Winkler**
   - **Fonético** (Double Metaphone)
   - **Embeddings** (SentenceTransformers)
5) Evaluar contra etiquetas (`labels_sinteticos.csv`) si están disponibles.


In [ ]:
import os
import re
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

from rapidfuzz.distance import Levenshtein
import jellyfish
from metaphone import doublemetaphone
from sentence_transformers import SentenceTransformer


## Configuración de rutas
Ajusta estas rutas si tu proyecto usa otra estructura.


In [ ]:
PROJECT_ROOT = Path(".").resolve()
DB_FILE = PROJECT_ROOT / "analytics.db"
SYNTH_CSV = PROJECT_ROOT / "data" / "terceros_sinteticos.csv"

# (Recomendado) Etiquetas para evaluación: mapea id_tercero -> id_registro (y opcionalmente case_type)
LABELS_CSV = PROJECT_ROOT / "data" / "labels_sinteticos.csv"

print("DB_FILE:", DB_FILE)
print("SYNTH_CSV:", SYNTH_CSV)
print("LABELS_CSV:", LABELS_CSV, "(exists:", LABELS_CSV.exists(), ")")


## Crear y cargar base sintética (terceros)
Debe tener este esquema:

`id_tercero,tipo_sujeto,nombres,apellidos,fecha_nacimiento,nacionalidad,numero_documento,tipo_documento,pais_residencia`


In [ ]:
!python -m pipeline.matching.generate_terceros

In [ ]:
df_t = pd.read_csv(SYNTH_CSV, dtype=str).fillna("")
df_t.head()


## Cargar `consolidado` desde SQLite

La tabla `consolidado` tiene campos tipo JSON en texto (`aliases`, `nacionalidad`).
Los convertimos a listas para facilitar matching.


In [ ]:
def json_list(x):
    if x is None:
        return []
    s = str(x).strip()
    if not s:
        return []
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else []
    except Exception:
        return []

conn = sqlite3.connect(DB_FILE)
df_c = pd.read_sql_query("""
SELECT
  id_registro,
  fuente,
  tipo_sujeto,
  nombres,
  apellidos,
  aliases,
  fecha_nacimiento,
  nacionalidad,
  numero_documento,
  tipo_sancion,
  fecha_sancion
FROM consolidado
""", conn)
conn.close()

df_c["aliases_list"] = df_c["aliases"].map(json_list)
df_c["nacionalidad_list"] = df_c["nacionalidad"].map(json_list)

for col in ["nombres","apellidos","numero_documento","tipo_sujeto","fuente","id_registro"]:
    df_c[col] = df_c[col].fillna("").astype(str)

df_c.head()


## Normalización de texto
Usamos una normalización simple para el notebook (upper, strip, colapsar espacios).


In [ ]:
def norm(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip().upper()
    s = re.sub(r"\s+", " ", s)
    return s

def full_name(nombres: str, apellidos: str) -> str:
    n = norm(nombres)
    a = norm(apellidos)
    if n and a:
        return f"{n} {a}"
    return n or a or ""

df_t["full_name"] = [full_name(n, a) for n, a in zip(df_t["nombres"], df_t["apellidos"])]
df_c["full_name"] = [full_name(n, a) for n, a in zip(df_c["nombres"], df_c["apellidos"])]

df_t[["id_tercero","tipo_sujeto","full_name","numero_documento"]].head()


## Blocking (reducción de candidatos)

- Si el tercero trae `numero_documento`, primero buscamos candidatos por documento exacto.
- Si no, hacemos blocking por `tipo_sujeto` + prefijo de 3 letras del nombre.

Esto es crítico para escalar.


In [ ]:
def build_blocks_consolidado(df_c: pd.DataFrame) -> dict:
    blocks = {}

    doc_map = {}
    for idx, row in df_c.iterrows():
        doc = norm(row.get("numero_documento",""))
        if doc:
            doc_map.setdefault(doc, []).append(idx)

    pref_map = {}
    for idx, row in df_c.iterrows():
        tipo = (row.get("tipo_sujeto","") or "").strip().upper()
        name = norm(row.get("full_name",""))
        pref = name[:3] if len(name) >= 3 else name
        if tipo and pref:
            pref_map.setdefault((tipo, pref), []).append(idx)

    blocks["doc_map"] = doc_map
    blocks["pref_map"] = pref_map
    return blocks

blocks = build_blocks_consolidado(df_c)
len(blocks["doc_map"]), len(blocks["pref_map"])


In [ ]:
def get_candidate_indices(tercero_row: pd.Series, blocks: dict, max_candidates: int = 2000) -> list:
    tipo = (tercero_row.get("tipo_sujeto","") or "").strip().upper()
    doc = norm(tercero_row.get("numero_documento",""))
    name = norm(tercero_row.get("full_name",""))
    pref = name[:3] if len(name) >= 3 else name

    if doc and doc in blocks["doc_map"]:
        return blocks["doc_map"][doc][:max_candidates]

    if tipo and pref and (tipo, pref) in blocks["pref_map"]:
        return blocks["pref_map"][(tipo, pref)][:max_candidates]

    return []


## Modelos de similitud
Calculamos score (0–1):
- Levenshtein: `1 - dist/max_len`
- Jaro-Winkler: `jaro_winkler_similarity`
- Fonético: Double Metaphone (binario 0/1)


In [ ]:
def score_levenshtein(a: str, b: str) -> float:
    a, b = norm(a), norm(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    dist = Levenshtein.distance(a, b)
    m = max(len(a), len(b))
    return 1.0 - (dist / m) if m else 0.0

def score_jaro_winkler(a: str, b: str) -> float:
    a, b = norm(a), norm(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return float(jellyfish.jaro_winkler_similarity(a, b))

def score_phonetic(a: str, b: str) -> float:
    a, b = norm(a), norm(b)
    if not a or not b:
        return 0.0
    a_codes = [c for c in doublemetaphone(a) if c]
    b_codes = [c for c in doublemetaphone(b) if c]
    if not a_codes or not b_codes:
        return 0.0
    return 1.0 if set(a_codes).intersection(set(b_codes)) else 0.0


## Embeddings (SentenceTransformers)

- Precalculamos embeddings del consolidado (nombre principal + aliases)
- Para cada tercero, embed y cosine similarity (dot product porque normalizamos)


In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(MODEL_NAME)

def build_consolidado_text(row: pd.Series) -> str:
    base = norm(row.get("full_name",""))
    aliases = row.get("aliases_list", [])
    aliases = [norm(a) for a in aliases if a]
    if aliases:
        return base + " || " + " || ".join(aliases[:5])
    return base

df_c["text_for_emb"] = df_c.apply(build_consolidado_text, axis=1)
df_c["text_for_emb"].head()


In [ ]:
c_texts = df_c["text_for_emb"].tolist()
c_emb = emb_model.encode(c_texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
c_emb.shape


In [ ]:
def embed_texts(texts: list[str]) -> np.ndarray:
    return emb_model.encode(texts, batch_size=256, show_progress_bar=False, normalize_embeddings=True)


## Ranking Top-K por tercero


In [ ]:
def rank_candidates_for_tercero(tercero_row: pd.Series, top_k: int = 5) -> pd.DataFrame:
    cand_idxs = get_candidate_indices(tercero_row, blocks, max_candidates=2000)
    if not cand_idxs:
        return pd.DataFrame(columns=[
            "id_tercero","id_registro","fuente","full_name",
            "score_lev","score_jw","score_phon","score_emb"
        ])

    t_name = tercero_row.get("full_name","")
    t_doc = norm(tercero_row.get("numero_documento",""))
    t_text = norm(t_name)

    t_emb = embed_texts([t_text])[0]

    rows = []
    for idx in cand_idxs:
        c = df_c.iloc[idx]
        c_name = c.get("full_name","")

        doc_boost = 0.15 if (t_doc and t_doc == norm(c.get("numero_documento",""))) else 0.0

        s_lev = min(1.0, score_levenshtein(t_name, c_name) + doc_boost)
        s_jw  = min(1.0, score_jaro_winkler(t_name, c_name) + doc_boost)
        s_ph  = min(1.0, score_phonetic(t_name, c_name) + doc_boost)
        s_emb = min(1.0, float(np.dot(c_emb[idx], t_emb)) + doc_boost)

        rows.append({
            "id_tercero": tercero_row["id_tercero"],
            "id_registro": c["id_registro"],
            "fuente": c["fuente"],
            "full_name": c_name,
            "score_lev": s_lev,
            "score_jw": s_jw,
            "score_phon": s_ph,
            "score_emb": s_emb,
        })

    out = pd.DataFrame(rows)
    out = out.sort_values("score_emb", ascending=False).head(top_k)
    return out


## Demo rápido


In [ ]:
sample = df_t.sample(1, random_state=42).iloc[0]
sample


In [ ]:
rank_candidates_for_tercero(sample, top_k=10)


## Evaluación (si tienes `labels_sinteticos.csv`)

Formato recomendado:
- `id_tercero`
- `id_registro_target`
- `case_type` (opcional)


In [ ]:
def evaluate_hit_at_k(preds: pd.DataFrame, labels: pd.DataFrame, k: int, score_col: str) -> pd.DataFrame:
    preds2 = preds.sort_values(["id_tercero", score_col], ascending=[True, False]).copy()
    preds2["rank"] = preds2.groupby("id_tercero").cumcount() + 1
    topk = preds2[preds2["rank"] <= k].copy()

    merged = topk.merge(labels, on="id_tercero", how="inner")
    merged["hit"] = (merged["id_registro"] == merged["id_registro_target"]).astype(int)

    hit_by_t = merged.groupby("id_tercero")["hit"].max().reset_index()
    hit_at_k = float(hit_by_t["hit"].mean()) if len(hit_by_t) else 0.0

    hits = merged[merged["hit"] == 1].copy()
    if len(hits) == 0:
        mrr = 0.0
    else:
        first_rank = hits.groupby("id_tercero")["rank"].min()
        mrr = float((1.0 / first_rank).mean())

    return pd.DataFrame([{
        "model": score_col,
        "k": k,
        "hit_at_k": hit_at_k,
        "mrr": mrr,
        "n_eval": int(labels["id_tercero"].nunique()) if "id_tercero" in labels else len(labels)
    }])

if LABELS_CSV.exists():
    labels = pd.read_csv(LABELS_CSV, dtype=str)
    if "id_registro_target" not in labels.columns and "id_registro" in labels.columns:
        labels = labels.rename(columns={"id_registro": "id_registro_target"})
    labels.head()
else:
    print("No existe labels_sinteticos.csv todavía. Puedes crearlo para evaluación.")


In [ ]:
# Generar predicciones para todos los terceros (top 10)
TOP_K = 10

pred_rows = []
for _, trow in df_t.iterrows():
    ranked = rank_candidates_for_tercero(trow, top_k=TOP_K)
    if len(ranked) == 0:
        continue
    pred_rows.append(ranked)

preds_all = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
preds_all.head()


In [ ]:
if LABELS_CSV.exists() and len(preds_all) > 0:
    labels = pd.read_csv(LABELS_CSV, dtype=str)
    if "id_registro_target" not in labels.columns and "id_registro" in labels.columns:
        labels = labels.rename(columns={"id_registro": "id_registro_target"})

    metrics = pd.concat([
        evaluate_hit_at_k(preds_all, labels, k=10, score_col="score_lev"),
        evaluate_hit_at_k(preds_all, labels, k=10, score_col="score_jw"),
        evaluate_hit_at_k(preds_all, labels, k=10, score_col="score_phon"),
        evaluate_hit_at_k(preds_all, labels, k=10, score_col="score_emb"),
    ], ignore_index=True)

    metrics.sort_values("hit_at_k", ascending=False)
else:
    print("Falta labels_sinteticos.csv o preds_all vacío.")
